# Makemore Part 1 — Bigram Language Model

This notebook builds a **bigram character-level language model** that learns to generate human-like names by studying patterns in a large list of real names.

We implement the model in **two ways** and compare them:

| Approach | Method | Key idea |
|---|---|---|
| **Statistical** | Count matrix + smoothing | Directly count how often each pair of characters appears |
| **Neural Network** | Single linear layer + softmax | Learn the same distribution via gradient descent |

Both approaches ultimately optimise the same objective — **minimising the average negative log-likelihood** of the training data — and produce equivalent results.

**Topics covered:**
- Character tokenisation and bigram extraction
- Probability estimation with Laplace (add-1) smoothing
- One-hot encoding and the softmax function
- Forward / backward pass and gradient descent
- L2 regularisation
- Autoregressive sampling

## 1  Imports

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

%matplotlib inline

## 2  Load and Explore the Dataset

`names.txt` contains one name per line (e.g. *emma*, *olivia*, …).  
We read every line into a Python list so that each element is a single lowercase name string.

In [ ]:
words = open('names.txt', 'r').read().splitlines()

print(f'Total words : {len(words)}')
print(f'Shortest    : {min(len(w) for w in words)} characters')
print(f'Longest     : {max(len(w) for w in words)} characters')
print(f'First 10    : {words[:10]}')

## 3  Build the Vocabulary and Character Mappings

A **bigram model** works at the character level.  Every character needs an integer index so we can store counts in a matrix.

- `stoi` (string → int): maps each character to its row/column index.  
- `itos` (int → string): the reverse lookup used when decoding predictions.  
- The special token `.` (index **0**) serves as both a start-of-word and end-of-word marker, so the model learns how names begin and end.

In [ ]:
chars = sorted(list(set(''.join(words))))   # all unique characters in the dataset
stoi  = {s: i + 1 for i, s in enumerate(chars)}   # 'a'->1, 'b'->2, ..., 'z'->26
stoi['.'] = 0                                       # special boundary token
itos  = {i: s for s, i in stoi.items()}             # reverse map

vocab_size = len(stoi)
print(f'Vocabulary size: {vocab_size}')             # should be 27 (26 letters + '.')
print(f'Sample mapping : {list(stoi.items())[:5]}')

## 4  Build the Bigram Count Matrix

We iterate over every consecutive pair of characters in each word (including the boundary `.` tokens) and tally the counts in a **27 × 27** integer matrix `N`.  

`N[i, j]` = number of times character `i` was immediately followed by character `j` in the training set.

In [ ]:
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)

for w in words:
    chs = ['.'] + list(w) + ['.']      # wrap word with boundary tokens
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        N[ix1, ix2] += 1

print('Bigram count matrix shape:', N.shape)
print('Most common bigrams (top 5):')
flat_counts = [(itos[i] + itos[j], N[i, j].item())
               for i in range(vocab_size) for j in range(vocab_size)]
for bigram, count in sorted(flat_counts, key=lambda x: -x[1])[:5]:
    print(f'  {bigram!r:6s}  {count}')

## 5  Visualise the Bigram Count Matrix

A heatmap makes the count matrix easy to interpret visually.  
- **Rows** = the *current* character (context)  
- **Columns** = the *next* character (prediction)  
- Darker blue cells indicate more frequent bigrams.

In [ ]:
plt.figure(figsize=(16, 16))
plt.imshow(N, cmap='Blues')
for i in range(vocab_size):
    for j in range(vocab_size):
        chstr = itos[i] + itos[j]
        plt.text(j, i, chstr,   ha='center', va='bottom', color='gray', fontsize=8)
        plt.text(j, i, N[i, j].item(), ha='center', va='top',    color='gray', fontsize=7)
plt.title('Bigram Count Matrix', fontsize=16)
plt.axis('off')
plt.tight_layout()

## 6  Statistical Model: Probability Matrix with Laplace Smoothing

To turn raw counts into probabilities we:
1. **Add 1 to every count** (Laplace / add-one smoothing) — this avoids zero probabilities for unseen bigrams, which would blow up the log-likelihood to −∞.
2. **Normalise each row** so all 27 values sum to 1, giving `P[i]` the probability distribution over the next character given character `i`.

In [ ]:
P = (N + 1).float()                          # add-one smoothing
P /= P.sum(dim=1, keepdim=True)              # normalise each row → row-wise probability dist.

print('Row 0 (probabilities for next char after "."):')
print(P[0])
print(f'Row sum check: {P[0].sum().item():.4f}')   # should be 1.0

## 7  Sample Names from the Statistical Model

To generate a name we do **autoregressive sampling**:
1. Start with the boundary token `.` (index 0).
2. Look up the probability distribution for the current character: `P[ix]`.
3. Sample the next character index using `torch.multinomial`.
4. Append the corresponding character to the output.
5. Repeat until the `.` end token is sampled.

In [ ]:
g = torch.Generator().manual_seed(2147483647)   # fixed seed for reproducibility

print('Names sampled from the statistical bigram model:')
for _ in range(10):
    out = []
    ix = 0
    while True:
        p  = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:      # end token reached → word is complete
            break
    print(''.join(out[:-1]))   # strip trailing '.'

## 8  Evaluating the Statistical Model: Negative Log-Likelihood

We need a scalar **loss** to measure model quality.  We use the **average negative log-likelihood (NLL)**:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^{N} \log P(c_{i+1} \mid c_i)$$

- A **lower** NLL means the model assigns higher probability to the actual next characters → it has learned the data better.
- Taking the log converts the product of probabilities into a sum (numerically stable).
- Negating flips the sign so we *minimise* rather than maximise.

In [ ]:
log_likelihood = 0.0
n = 0

for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        ix1 = stoi[ch1]
        ix2 = stoi[ch2]
        log_likelihood += torch.log(P[ix1, ix2]).item()
        n += 1

nll = -log_likelihood
avg_nll = nll / n

print(f'Total log-likelihood : {log_likelihood:.2f}')
print(f'Total NLL            : {nll:.2f}')
print(f'Average NLL (loss)   : {avg_nll:.4f}')

## 9  Neural Network Approach: Dataset Preparation

We now replicate the statistical model using a **single linear layer** trained with gradient descent.  The key insight is:

> The rows of the weight matrix `W` after training should converge to (log) values proportional to the rows of the count matrix `N`.

**Dataset format:**  
- `xs` — integer tensor of *input* character indices (one per bigram)  
- `ys` — integer tensor of *target* character indices (the next character for each input)

We extract bigrams from the **full** dataset, so both tensors have one entry per consecutive character pair across all words.

In [ ]:
xs, ys = [], []
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)   # shape: (num_bigrams,)
ys = torch.tensor(ys)   # shape: (num_bigrams,)

num = xs.nelement()
print(f'Total training bigrams: {num}')
print(f'xs[:5] = {xs[:5].tolist()}  →  {[itos[i] for i in xs[:5].tolist()]}')
print(f'ys[:5] = {ys[:5].tolist()}  →  {[itos[i] for i in ys[:5].tolist()]}')

## 10  One-Hot Encoding

The neural network expects a **continuous vector** as input, not an integer index.  
We convert each integer index to a **one-hot vector** of length 27:  
all zeros except a single `1` at position `i`.

After encoding, `xenc` has shape `(num_bigrams, 27)` — one row per training example.

In [ ]:
xenc = F.one_hot(xs, num_classes=vocab_size).float()   # (num_bigrams, 27)

print('One-hot encoded input shape:', xenc.shape)
print('First example (input char index 0 = "."):')  
plt.figure(figsize=(12, 2))
plt.imshow(xenc[:5], cmap='Greys', aspect='auto')
plt.xlabel('Character index')
plt.ylabel('Example')
plt.title('First 5 one-hot encoded inputs')
plt.colorbar()
plt.tight_layout()

## 11  Initialise the Neural Network

Our model is a **single linear layer** — just one weight matrix `W` of shape `(27, 27)`.  
There is no bias and no activation function before the softmax.

- The matrix multiplication `xenc @ W` produces **logits** — unnormalised log-counts.  
- `W` is initialised with random normal values and tagged with `requires_grad=True` so PyTorch can compute gradients during backprop.

Conceptually this is equivalent to learning `log N[i]` (up to normalisation) for each row `i`.

In [ ]:
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

print('Weight matrix shape:', W.shape)
print('(Each of the 27 rows is the learned log-count distribution for one input character.)')

## 12  Training Loop — Forward Pass, Backward Pass, and Weight Update

Each iteration of gradient descent performs three steps:

1. **Forward pass** — compute the loss:
   - `logits = xenc @ W` — dot product gives one logit vector per example
   - `counts = logits.exp()` — exponentiate to get positive "pseudo-counts"
   - `probs = counts / counts.sum(dim=1, keepdim=True)` — normalise → softmax probabilities
   - `loss` = mean NLL of the correct next characters + L2 regularisation on `W`

2. **Backward pass** — `loss.backward()` computes ∂loss/∂W automatically.

3. **Parameter update** — gradient descent: `W ← W - lr × ∂loss/∂W`.

> **L2 regularisation** (`0.01 * (W**2).mean()`) penalises large weights, acting as a smoothing prior that prevents the model from becoming overconfident on rare bigrams.

In [ ]:
# Re-initialise weights so this cell can be run standalone
g = torch.Generator().manual_seed(2147483647)
W = torch.randn((vocab_size, vocab_size), generator=g, requires_grad=True)

learning_rate = 50
num_steps     = 200
reg_strength  = 0.01     # L2 regularisation coefficient

losses = []
for step in range(num_steps):

    # ── Forward pass ──────────────────────────────────────────────────
    xenc   = F.one_hot(xs, num_classes=vocab_size).float()  # (N, 27)
    logits = xenc @ W                                        # (N, 27) — log-counts
    counts = logits.exp()                                    # (N, 27) — pseudo-counts
    probs  = counts / counts.sum(dim=1, keepdim=True)        # (N, 27) — softmax probs

    # NLL of correct next characters + L2 regularisation
    loss = -probs[torch.arange(num), ys].log().mean() + reg_strength * (W ** 2).mean()
    losses.append(loss.item())

    # ── Backward pass ─────────────────────────────────────────────────
    W.grad = None          # reset gradients (equivalent to optimizer.zero_grad())
    loss.backward()

    # ── Weight update (gradient descent) ──────────────────────────────
    W.data -= learning_rate * W.grad

    if step % 20 == 0 or step == num_steps - 1:
        print(f'Step {step:4d} | Loss: {loss.item():.4f}')

# Plot training curve
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Gradient descent step')
plt.ylabel('Loss (Avg NLL + L2 reg)')
plt.title('Training Loss')
plt.grid(True)
plt.tight_layout()

## 13  Sample Names from the Neural Network Model

Sampling from the neural network follows the same autoregressive procedure as before, but instead of looking up a row in the pre-computed probability matrix `P`, we:
1. One-hot encode the current character index.
2. Do a forward pass through `W` to get logits.
3. Apply softmax to obtain probabilities.
4. Sample the next character index from that distribution.

Because the neural net has been trained to minimise the same NLL objective, the generated names should look similar to those from the statistical model.

In [ ]:
g = torch.Generator().manual_seed(2147483647)

print('Names sampled from the trained neural network:')
for _ in range(10):
    out = []
    ix  = 0
    while True:
        # One forward pass for a single character
        xenc   = F.one_hot(torch.tensor([ix]), num_classes=vocab_size).float()  # (1, 27)
        logits = xenc @ W                                                         # (1, 27)
        counts = logits.exp()
        p      = counts / counts.sum(dim=1, keepdim=True)                         # softmax

        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        out.append(itos[ix])
        if ix == 0:   # end token → word complete
            break
    print(''.join(out[:-1]))

## 14  Comparison: Statistical vs. Neural Network Model

Both approaches compute the same kind of probability distribution over the next character.  
Here we verify that their average NLLs are close — confirming the neural network has converged to approximately the same solution as the statistical model.

In [ ]:
# ── Statistical model NLL (from Section 8) ────────────────────────────────
log_lik_stat = 0.0
n_stat       = 0
for w in words:
    chs = ['.'] + list(w) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        log_lik_stat += torch.log(P[stoi[ch1], stoi[ch2]]).item()
        n_stat += 1
nll_stat = -log_lik_stat / n_stat

# ── Neural network model NLL ───────────────────────────────────────────────
with torch.no_grad():
    xenc_all = F.one_hot(xs, num_classes=vocab_size).float()
    logits   = xenc_all @ W
    counts   = logits.exp()
    probs    = counts / counts.sum(dim=1, keepdim=True)
    nll_nn   = -probs[torch.arange(num), ys].log().mean().item()

print(f'Statistical model  — Average NLL: {nll_stat:.4f}')
print(f'Neural network     — Average NLL: {nll_nn:.4f}')
print()
print('The two values converge because the neural network learns the same')
print('log-count statistics that the count matrix captures directly.')